## Debugging

### Syntax Errors

In [1]:
for i in range(5)   # missing colon
    print(i)

SyntaxError: expected ':' (ipython-input-1262323080.py, line 1)

In [3]:
x = 11
if x > 10:
  x=10
print(x)            # wrong indentation

10


### Runtime Errors

In [5]:
# Name Error
if y < 10:
  print(y)

NameError: name 'y' is not defined

In [6]:
# Sample DataFrame
import pandas as pd
data = {'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Score': [85, 92, 78, 80]}
df = pd.DataFrame(data, index=[3, 1, 4, 2])

In [7]:
# Key Error
df["Subject"]



KeyError: 'Subject'

In [8]:
# Index Error
df.iloc[10]

IndexError: single positional indexer is out-of-bounds

In [9]:
class DatasetStats:
    """
    Simple statistics helper class for a list of values.
    """

    def __init__(self, values):
        values = list(values)
        if len(values) == 0:
            raise ValueError("values must not be empty")
        self.values = values

    def mean(self):
        return sum(self.values) / len(self.values)

    def std(self):
        m = self.mean()
        var = sum((x - m) ** 2 for x in self.values) / len(self.values)
        return var ** 0.5

    def z_scores(self):
        m = self.mean()
        s = self.std()
        if s == 0:
            return [0.0 for _ in self.values]
        return [(x - m) / s for x in self.values]

In [14]:
data_stats1 = DatasetStats([1, 2, 3, 4])
data_stats1.mean()

2.5

In [16]:
# Type Error
"abc" + 10

TypeError: can only concatenate str (not "int") to str

### Effective Debugging

In [17]:
%pdb on

Automatic pdb calling has been turned ON


In [23]:
def compute_discounted_price(price, discount_pct):
    return price * (1 - discount_pct / 100)


def compute_cart_total(cart):
    total = 0
    for item in cart:
        # if "discount" not in item:
        #     print(f"Item with missing discount:{item["name"]}")
        price = item["price"]
        if "discount" not in item:
            discount = 0
        else:
            discount = item["discount"]
        total += compute_discounted_price(price, discount)
    return total


cart = [
    {"name": "Shirt", "price": 1000, "discount": 10},
    {"name": "Pants", "price": 1500, "discount": 20},
    {"name": "Socks", "price": 200}
]

def demo():
    print("Cart total:", compute_cart_total(cart))

demo()


Cart total: 2300.0


In [24]:
%debug

> /usr/local/lib/python3.12/dist-packages/IPython/core/compilerop.py(101)ast_parse()
     99         Arguments are exactly the same as ast.parse (in the standard library),
    100         and are passed to the built-in compile function."""
--> 101         return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)
    102 
    103     def reset_compiler_flags(self):

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user


## Testing using PyTest

In [1]:
!pip install pytest


### Test Code 1

In [2]:
# Demo function
# Below command writes the contents of the cell to a file
%%writefile mymath.py
def add(a, b):
    return a + b


Writing mymath.py


In [6]:
%%writefile test_mymath.py

from mymath import add

def test_add_positive():
    assert add(2, 3) == 5

def test_add_negative():
    assert add(-1, -1) == -2

# def test_add_zero():
#     assert add(0, 5) == 5


Overwriting test_mymath.py


In [7]:
!pytest -v test_mymath.py


============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.4.47, anyio-4.11.0, typeguard-4.4.4
collected 2 items                                                              

test_mymath.py::test_add_positive PASSED                                 [ 50%]
test_mymath.py::test_add_negative PASSED                                 [100%]

============================== 2 passed in 0.01s ===============================


### Test Code 2 using PyTest Fixtures

In [8]:
%%writefile data_processing.py
import pandas as pd


def clean_age_column(df, col = "age"):
    """
    Drop rows with missing age, convert to int, and ensure no negative ages
    """
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found")

    cleaned = df.dropna(subset=[col]).copy()
    cleaned[col] = cleaned[col].astype(int)

    if (cleaned[col] < 0).any():
        raise ValueError("Age cannot be negative")

    return cleaned


def filter_valid_scores(df, col = "score"):
    """
    Keep only rows where score is between 0 and 100 (inclusive).
    Returns a NEW filtered DataFrame.
    """
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found")

    mask = df[col].between(0, 100)
    return df.loc[mask].copy()


Writing data_processing.py


In [16]:
%%writefile test_data_processing.py

import numpy as np
import pandas as pd
import pytest

from data_processing import clean_age_column, filter_valid_scores

@pytest.fixture
def sample_df():
    return pd.DataFrame({
        "user_id": [1, 2, 3],
        "age": ["20", "30", None],
        "score": [90, 150, 80],
    })


@pytest.fixture
def df_with_negative_age():
    return pd.DataFrame({
        "user_id": [10, 11],
        "age": ["10", "-5"],
        "score": [50, 60],
    })


def test_clean_age(sample_df):
    cleaned = clean_age_column(sample_df, col="age")

    # Missing age row should be dropped → 2 rows left
    assert cleaned.shape[0] == 2


def test_valid_scores(sample_df):
    filtered = filter_valid_scores(sample_df, col="score")

    # All scores should be within [0, 100]
    assert filtered["score"].between(0, 100).all()

    # The row with score 150 should be removed
    assert filtered.shape[0] == 2

Overwriting test_data_processing.py


In [17]:
!pytest -v test_data_processing.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.4.47, anyio-4.11.0, typeguard-4.4.4
collected 2 items                                                              

test_data_processing.py::test_clean_age PASSED                           [ 50%]
test_data_processing.py::test_valid_scores PASSED                        [100%]

============================== 2 passed in 0.57s ===============================


### Test Code 3 using PyTest Parameterisation

In [21]:
%%writefile normalise.py

def normalize_scores(scores):
    """
    Min-max normalize a list of numeric scores to [0, 1].
    If all scores are the same, returns all zeros.
    """
    scores = list(scores)
    if len(scores) == 0:
        return []

    min_s = min(scores)
    max_s = max(scores)

    # if max_s == min_s:
    #     return [0.0 for _ in scores]

    return [(s - min_s) / (max_s - min_s) for s in scores]

Overwriting normalise.py


In [22]:
%%writefile test_normalise.py
import pytest
from normalise import normalize_scores

# Decorator in pytest
@pytest.mark.parametrize("scores, expected",
    [
        ([0, 5, 10], [0.0, 0.5, 1.0]),
        ([10, 10, 10], [0.0, 0.0, 0.0]),
        ([], []),
    ],
)
def test_normalize_scores(scores, expected):
    result = normalize_scores(scores)
    assert pytest.approx(result) == expected

Overwriting test_normalise.py


In [23]:
!pytest -v test_normalise.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.4.47, anyio-4.11.0, typeguard-4.4.4
collected 3 items                                                              

test_normalise.py::test_normalize_scores[scores0-expected0] PASSED       [ 33%]
test_normalise.py::test_normalize_scores[scores1-expected1] FAILED       [ 66%]
test_normalise.py::test_normalize_scores[scores2-expected2] PASSED       [100%]

=================================== FAILURES ===================================
___________________ test_normalize_scores[scores1-expected1] ___________________

scores = [10, 10, 10], expected = [0.0, 0.0, 0.0]

    @pytest.mark.parametrize("scores, expected",
        [
            ([0, 5, 10], [0.0, 0.5, 1.0]),
            ([10, 10, 10], [0.0, 0.0, 0.0]),
            ([], []),
        ],
    )
    def test_norm

### Test Code 4 using Pytest Approx

In [24]:
%%writefile data_stats.py

class DatasetStats:
    """
    Simple statistics helper class for a list of values.
    """

    def __init__(self, values):
        values = list(values)
        if len(values) == 0:
            raise ValueError("values must not be empty")
        self.values = values

    def mean(self):
        return sum(self.values) / len(self.values)

    def std(self):
        m = self.mean()
        var = sum((x - m) ** 2 for x in self.values) / len(self.values)
        return var ** 0.5

    def z_scores(self):
        m = self.mean()
        s = self.std()
        if s == 0:
            return [0.0 for _ in self.values]
        return [(x - m) / s for x in self.values]

Writing data_stats.py


In [25]:
%%writefile test_stats.py

import pytest
from data_stats import DatasetStats

def test_dataset_stats_mean_std():
    stats = DatasetStats([1, 2, 3, 4])
    assert stats.mean() == 2.5
    # variance = ((1.5^2 + 0.5^2 + 0.5^2 + 1.5^2)/4) = 1.25, std = sqrt(1.25)
    assert pytest.approx(stats.std()) == (1.25 ** 0.5)


def test_dataset_stats_z_scores_zero_variance():
    stats = DatasetStats([5, 5, 5])
    zs = stats.z_scores()
    # If std = 0, we return all zeros
    assert all(z == 0.0 for z in zs)

Writing test_stats.py


In [26]:
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.4.47, anyio-4.11.0, typeguard-4.4.4
collected 9 items                                                              

test_data_processing.py::test_clean_age PASSED                           [ 11%]
test_data_processing.py::test_valid_scores PASSED                        [ 22%]
test_mymath.py::test_add_positive PASSED                                 [ 33%]
test_mymath.py::test_add_negative PASSED                                 [ 44%]
test_normalise.py::test_normalize_scores[scores0-expected0] PASSED       [ 55%]
test_normalise.py::test_normalize_scores[scores1-expected1] FAILED       [ 66%]
test_normalise.py::test_normalize_scores[scores2-expected2] PASSED       [ 77%]
test_stats.py::test_dataset_stats_mean_std PASSED                        [ 88%]
test_stats.py::tes

### Test Code 5 testing exception

In [27]:
%%writefile test_negative_age.py

import pytest
from data_processing import clean_age_column
from test_data_processing import df_with_negative_age

def test_negative_age(df_with_negative_age):
    with pytest.raises(ValueError):
        clean_age_column(df_with_negative_age, col="age")

Writing test_negative_age.py


In [29]:
!pytest -v test_negative_age.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.4.47, anyio-4.11.0, typeguard-4.4.4
collected 1 item                                                               

test_negative_age.py::test_negative_age PASSED                           [100%]

============================== 1 passed in 0.55s ===============================


## Type Hints and Data Validation

### Types Hints

In [30]:
# Type hints applied to a function

from typing import List, Tuple

def scale(values : List[float]) -> List[float]:
    max_val = max(values)
    return [v / max_val for v in values]

In [31]:
scale([50, 40, 10, 5])

[1.0, 0.8, 0.2, 0.1]

In [46]:
# Type hints applied to a function

import pandas as pd

def clean_age_column(df: pd.DataFrame, col = "age") -> pd.DataFrame:
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found")

    cleaned = df.dropna(subset=[col]).copy()
    cleaned[col] = cleaned[col].astype(int)

    if (cleaned[col] < 0).any():
        raise ValueError("Age cannot be negative")

    return cleaned

### Data Validation using Pydantic

In [32]:
from pydantic import BaseModel, Field

class User(BaseModel):
    age: int = Field(gt=0)            # must be positive
    score: float = Field(ge=0, le=100)  # must be in [0, 100]


In [33]:
u1 = User(age=25, score=90)   # Satisfies the conditions


In [34]:
u2 = User(age=-5, score=150) # Throws an error

ValidationError: 2 validation errors for User
age
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
score
  Input should be less than or equal to 100 [type=less_than_equal, input_value=150, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal

In [36]:
# Validate a dataframe using pydantic
import pandas as pd

df = pd.DataFrame({
    "age": [25, -1, 30],
    "score": [90, 110, 80]
})


In [38]:
df

,age,score
0,25,90
1,-1,110
2,30,80


In [39]:
rows = df.to_dict(orient="records") # Row-wise dictionaries created

valid = []
invalid = []

for row in rows:
    try:
        print(row)
        valid.append(User(**row))
    except Exception as e:
        invalid.append((row, e))

print(invalid)

{'age': 25, 'score': 90}
{'age': -1, 'score': 110}
{'age': 30, 'score': 80}
[({'age': -1, 'score': 110}, 2 validation errors for User
age
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
score
  Input should be less than or equal to 100 [type=less_than_equal, input_value=110, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal)]


### Data Validation using Pandera

In [40]:
!pip install -q pandera

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.9/295.9 kB 6.1 MB/s eta 0:00:00


In [50]:
import pandas as pd
import numpy as np
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

schema = DataFrameSchema({
    "age": Column(int, Check.gt(0)),
    "score": Column(int, Check.between(0, 100)),
})

df = pd.DataFrame({
    "age": [25, -5, 30],
    "score": [90, 150, 80],
})

try:
    schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as e:
    print(e.failure_cases) # Returns the errors in the form of a dataframe

  schema_context column             check  check_number  failure_case  index
0         Column    age   greater_than(0)             0            -5      1
1         Column  score  in_range(0, 100)             0           150      1


In [52]:
df = pd.DataFrame({
    "age": [25, 40, 30],
    "score": [90, 150, np.nan],
})

schema = DataFrameSchema({
    "age": Column(int, Check.gt(0), nullable=False),
    "score": Column(float, Check.between(0, 100), nullable=True),
})
try:
    schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as e:
    print(e.failure_cases) # Returns the errors in the form of a dataframe

  schema_context column             check  check_number  failure_case  index
0         Column  score  in_range(0, 100)             0         150.0      1


### Pandera along with PyTest

In [63]:
%%writefile schemas.py
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema


# Schema for *input* data (raw)
raw_user_schema = DataFrameSchema({
    "user_id": Column(int),
    "age": Column(float, nullable=True),   # raw age may have missing
    "score": Column(float, nullable=True), # raw score may have missing
})


# Schema for *cleaned* data
clean_user_schema = DataFrameSchema({
    "user_id": Column(int),
    "age": Column(int, nullable=False),  # must be positive integer
    "score": Column(float, [
        Check.ge(0),
        Check.le(100),
    ], nullable=False),
})


Overwriting schemas.py


In [64]:
%%writefile data_processing.py
import pandas as pd

from schemas import raw_user_schema, clean_user_schema

def preprocess_users(df: pd.DataFrame) -> pd.DataFrame:
    # Validate the *input* DataFrame
    df = raw_user_schema.validate(df)

    # Drop missing values in key columns
    df = df.dropna(subset=["age", "score"]).copy()

    # Convert age to int
    df["age"] = df["age"].astype(int)

    # Clip score to [0, 100]
    df["score"] = df["score"].clip(lower=0, upper=100)

    # Validate the *output* DataFrame
    df = clean_user_schema.validate(df, lazy=True)

    return df


Overwriting data_processing.py


In [65]:
%%writefile test_data_processing.py
import pandas as pd
import pandera.pandas as pa
import pytest

from data_processing import preprocess_users
from schemas import clean_user_schema


def test_preprocess_users():
    df = pd.DataFrame({
        "user_id": [1, 2, 3],
        "age": [20.0, -30.0, 25.0],
        "score": [90.0, 80.0, 70.0],
    })

    result = preprocess_users(df)

    # # Validate against the clean schema
    # clean_user_schema.validate(result, lazy=True)

Overwriting test_data_processing.py


In [66]:
!pytest -v test_data_processing.py # --tb=no

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.4.47, anyio-4.11.0, typeguard-4.4.4
collected 1 item                                                               

test_data_processing.py::test_preprocess_users PASSED                    [100%]

============================== 1 passed in 1.27s ===============================
